In [1]:
import os
import csv
import chromadb

from pathlib import Path
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
import sys
print(sys.executable)

c:\Users\admin\AppData\Local\Programs\Python\Python314\python.exe


In [2]:
PROJECT_DIR = Path(r"D:\LLM TASKS\LLM_PDF_Question_Answering")

DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUT_DIR = DATASET_DIR
DB_PATH = PROJECT_DIR / "chroma_db"

DATASET_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH.mkdir(exist_ok=True)

PDF_CANDIDATES = [
    DATASET_DIR / "sample.pdf",
    DATASET_DIR / "sample(1).pdf"
]

PDF_PATH = next(
    (file for file in PDF_CANDIDATES if file.exists()),
    None
)

if PDF_PATH is None:
    raise FileNotFoundError(
        "PDF not found. Place sample.pdf or sample(1).pdf inside dataset folder."
    )

print("PDF File:", PDF_PATH)
print("Project paths configured successfully!")

PDF File: D:\LLM TASKS\LLM_PDF_Question_Answering\dataset\sample.pdf
Project paths configured successfully!


In [3]:
reader = PdfReader(str(PDF_PATH))

extracted_text = ""

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    extracted_text += page_text + "\n"

print("Total Pages:", len(reader.pages))
print("Extracted Characters:", len(extracted_text))
print("\nPDF Text Preview:\n")
print(extracted_text[:2000])

with open(
    OUTPUT_DIR / "extracted_text.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(extracted_text)

print("\nPDF text extracted and saved successfully!")

Total Pages: 379
Extracted Characters: 910384

PDF Text Preview:

Introduction to Large Language Models 
1 
 
 
Introduction to Large Language Models 
(LLMs) 
Principles, Architecture, and Applications 
 
 
Author 
Peretz Shoval 
 
 
 
September 2026 
  
Introduction to Large Language Models 
2 
Copyright 
Introduction to Large Language Models 
 
Copyright © Peretz Shoval. All rights reserved. 
 
Prepared with ChatGPT as a research, writing, and editorial assistant. 
 
 
 
 
 
 
 
 
This book is open for use by anyone who wants to learn or teach 
Large Language Models 
  
Introduction to Large Language Models 
3 
 
Table of Contents 
 
Preface 
Chapter 1 - Artificial Intelligence: Concepts, Evolution, and Foundations 
Chapter 2 - Machine Learning and Deep Learning 
Chapter 3 - Understanding Large Language Models 
Chapter 4 - Tokens, Embeddings and Context 
Chapter 5 - The Transformer Architecture 
Chapter 6 - Training Large Language Models 
Chapter 7 - Inference: How a Large Language M

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = text_splitter.split_text(extracted_text)

print("Total Chunks:", len(chunks))

with open(
    OUTPUT_DIR / "chunks.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)
    writer.writerow(["Chunk_ID", "Chunk_Text"])

    for index, chunk in enumerate(chunks):
        writer.writerow([index, chunk])

print("Text chunks created and saved successfully!")

Total Chunks: 1370
Text chunks created and saved successfully!


In [5]:
client = chromadb.PersistentClient(path=str(DB_PATH))

embedding_function = DefaultEmbeddingFunction()

collection = client.get_or_create_collection(
    name="week13_pdf_qa",
    embedding_function=embedding_function,
    metadata={"description": "Week 13 PDF Question Answering"}
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [6]:
BATCH_SIZE = 100

for start in range(0, len(chunks), BATCH_SIZE):

    batch_chunks = chunks[start:start + BATCH_SIZE]

    batch_ids = [
        f"chunk_{i}"
        for i in range(start, start + len(batch_chunks))
    ]

    batch_metadata = [
        {"chunk_number": i}
        for i in range(start, start + len(batch_chunks))
    ]

    collection.upsert(
        ids=batch_ids,
        documents=batch_chunks,
        metadatas=batch_metadata
    )

print("Total Stored Chunks:", collection.count())
print("Chunks stored successfully in ChromaDB!")

Total Stored Chunks: 2860
Chunks stored successfully in ChromaDB!


In [7]:
questions = [
    "What is Artificial Intelligence and how has it evolved?",
    "What are the different learning paradigms in Machine Learning?",
    "Why is data quality important in Machine Learning?",
    "What are neural networks and how do they work?",
    "What are tokens in Large Language Models?",
    "What are embeddings and why are they important?",
    "Explain the Transformer architecture.",
    "What is the role of self-attention in Transformers?",
    "How are Large Language Models trained?",
    "What happens during LLM inference?",
    "What are the capabilities and limitations of LLMs?",
    "What are the applications of LLMs in different industries?",
    "What is Retrieval-Augmented Generation (RAG)?",
    "What are the privacy, security and safety concerns of LLMs?",
    "Why are ethics and human oversight important in AI systems?"
]

with open(
    OUTPUT_DIR / "questions.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)
    writer.writerow(["Question"])
    for question in questions:
        writer.writerow([question])

print("Total Questions:", len(questions))
print("Questions saved successfully!")

Total Questions: 15
Questions saved successfully!


In [8]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print("Ollama LLM initialized successfully!")

Ollama LLM initialized successfully!


In [9]:
TOP_K = 5

search_results = []

for question in questions:

    results = collection.query(
        query_texts=[question],
        n_results=min(TOP_K, collection.count())
    )

    retrieved_documents = results["documents"][0]
    distances = results["distances"][0]

    for rank, (document, distance) in enumerate(
        zip(retrieved_documents, distances),
        start=1
    ):

        similarity_score = 1 / (1 + distance)

        search_results.append({
            "Question": question,
            "Rank": rank,
            "Retrieved_Document": document,
            "Distance": distance,
            "Similarity_Score": similarity_score
        })

print("Document retrieval completed!")
print("Total Retrieved Results:", len(search_results))

Document retrieval completed!
Total Retrieved Results: 75


In [10]:
with open(
    OUTPUT_DIR / "search_results.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "Question",
            "Rank",
            "Retrieved_Document",
            "Distance",
            "Similarity_Score"
        ]
    )

    writer.writeheader()
    writer.writerows(search_results)

print("Search results saved successfully!")

Search results saved successfully!


In [11]:
rag_answers = []

for question in questions:

    results = collection.query(
        query_texts=[question],
        n_results=min(TOP_K, collection.count())
    )

    context = "\n\n".join(results["documents"][0])

    prompt = f"""
    Answer the question using only the provided PDF context.

    If the answer is not available in the context,
    clearly mention that the information is not available
    in the provided PDF context.

    PDF Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    answer = response.content

    rag_answers.append(answer)

    print("Question:", question)
    print("RAG Answer:", answer)
    print("-" * 80)

print("RAG answer generation completed!")

Question: What is Artificial Intelligence and how has it evolved?
RAG Answer: Artificial Intelligence (AI) is a branch of computer science that involves designing systems to perform tasks normally associated with human intelligence, such as perception, reasoning, learning, planning, language understanding, decision making, and problem solving.

AI systems typically implement behavior through rules and logic explicitly specified by programmers, but modern machine-learning systems learn from data, analyzing large numbers of examples to discover statistical patterns that enable them to perform tasks with increasing accuracy.

AI is fundamentally different from human intelligence, providing no established evidence of consciousness, subjective experience, or self-awareness comparable to that of humans. Instead, its apparent intelligence arises from computational processes rather than human-like mental experience.

Artificial Intelligence did not emerge as a single invention, but rather evol

In [12]:
non_rag_answers = []

for question in questions:

    prompt = f"""
    Answer the following question using your general knowledge.

    Question:
    {question}

    Answer:
    """

    response = llm.invoke(prompt)

    answer = response.content

    non_rag_answers.append(answer)

    print("Question:", question)
    print("Non-RAG Answer:", answer)
    print("-" * 80)

print("Non-RAG answer generation completed!")

Question: What is Artificial Intelligence and how has it evolved?
Non-RAG Answer: Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception. The term "Artificial Intelligence" was first coined in 1956 by John McCarthy, a computer scientist and cognitive scientist.

The evolution of AI can be divided into several stages:

1. **Early Years (1950s-1970s)**: The first AI programs were developed in the 1950s and 1960s, focusing on rule-based systems and symbolic reasoning. These early systems were limited in their capabilities and were often referred to as "narrow" or "weak" AI.

2. **Rule-Based Expert Systems (1970s-1980s)**: In the 1970s and 1980s, AI research shifted towards developing rule-based expert systems, which mimicked human decision-making processes. These systems were used in various applications, such as medical diagnosis and fin

In [13]:
comparison_results = []

for index, question in enumerate(questions):

    comparison_results.append({
        "Question": question,
        "RAG_Answer": rag_answers[index],
        "Non_RAG_Answer": non_rag_answers[index]
    })

with open(
    OUTPUT_DIR / "answers_comparison.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "Question",
            "RAG_Answer",
            "Non_RAG_Answer"
        ]
    )

    writer.writeheader()
    writer.writerows(comparison_results)

print("RAG and Non-RAG comparison saved successfully!")

RAG and Non-RAG comparison saved successfully!


In [14]:
print("WEEK 13 - PDF QUESTION ANSWERING SYSTEM")
print("=" * 50)

print("PDF Name:", PDF_PATH.name)
print("Total PDF Pages:", len(reader.pages))
print("Total Extracted Characters:", len(extracted_text))
print("Total Text Chunks:", len(chunks))
print("Stored Chunks:", collection.count())
print("Total Questions:", len(questions))
print("RAG Answers:", len(rag_answers))
print("Non-RAG Answers:", len(non_rag_answers))

print("\nGenerated Files:")

for file in OUTPUT_DIR.iterdir():
    if file.is_file():
        print("-", file.name)

print("\nPDF Question Answering System completed successfully!")

WEEK 13 - PDF QUESTION ANSWERING SYSTEM
PDF Name: sample.pdf
Total PDF Pages: 379
Total Extracted Characters: 910384
Total Text Chunks: 1370
Stored Chunks: 2860
Total Questions: 15
RAG Answers: 15
Non-RAG Answers: 15

Generated Files:
- answers_comparison.csv
- chunks.csv
- extracted_text.txt
- questions.csv
- sample.pdf
- search_results.csv

PDF Question Answering System completed successfully!
